In [0]:
df= spark.table("ecommerce_db.gold.category_daily")

In [0]:
display(df.head(5))

category_code,event_date,views,carts,purchases
appliances.kitchen.steam_cooker,2019-10-14,170,7,4
computers.peripherals.camera,2019-10-26,69,1,1
computers.components.memory,2019-11-08,539,46,15
kids.dolls,2019-10-19,498,6,8
sport.bicycle,2019-11-17,2275,179,121


In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Train/test split
train, test = df.randomSplit([0.8, 0.2], seed=42)

# assemling all the features 
assembler = VectorAssembler(inputCols=["views", "carts"], outputCol="features")

#  Models to compare (Spark ML)
models = {
    "LinearRegression": LinearRegression(featuresCol="features", labelCol="purchases"),
    "DecisionTree": DecisionTreeRegressor(featuresCol="features", labelCol="purchases", maxDepth=5),
    "RandomForest": RandomForestRegressor(featuresCol="features", labelCol="purchases", numTrees=100, maxDepth=5)
}

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_eval = RegressionEvaluator(labelCol="purchases",predictionCol="prediction",metricName="rmse")
r2_eval = RegressionEvaluator(labelCol="purchases",predictionCol="prediction",metricName="r2")

In [0]:
import mlflow
import mlflow.spark

mlflow.set_experiment("/Users/niharikaganji22@gmail.com/databricks indian data club/DAY 13 Model Comparison & Feature Engineering")

for name, algo in models.items():
    with mlflow.start_run(run_name=name):

        # Build pipeline = feature step + model
        pipeline = Pipeline(stages=[assembler, algo])

        # Train
        pipeline_model = pipeline.fit(train)

        # Predict
        preds = pipeline_model.transform(test)

        # Evaluate
        rmse = rmse_eval.evaluate(preds)
        r2 = r2_eval.evaluate(preds)

        # Log parameters
        mlflow.log_param("model_type", name)
        mlflow.log_param("features", "views,carts")

        # Log metrics
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)

        # Log full pipeline model
        mlflow.spark.log_model(pipeline_model,"model", dfs_tmpdir="/Volumes/workspace/ecommerce/ecommerce_data")
        print(f"{name} → RMSE: {rmse:.2f}, R²: {r2:.4f}")

2026/01/21 17:31:39 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.2.2) contains a local version label (+databricks.connect.17.2.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/01/21 17:31:43 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-8a47de59-9c0f-4f3e-8889-b1/tmpk88dap86/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/01/21 17:31:43 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


LinearRegression → RMSE: 129.47, R²: 0.9068


2026/01/21 17:32:21 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.2.2) contains a local version label (+databricks.connect.17.2.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/01/21 17:32:24 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-8a47de59-9c0f-4f3e-8889-b1/tmp6p34z7uu/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/01/21 17:32:24 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


DecisionTree → RMSE: 426.68, R²: 0.4807


2026/01/21 17:33:08 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.2.2) contains a local version label (+databricks.connect.17.2.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/01/21 17:33:11 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-8a47de59-9c0f-4f3e-8889-b1/tmp9soj9vij/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/01/21 17:33:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


RandomForest → RMSE: 399.71, R²: 0.4775


In [0]:
import mlflow
runs = mlflow.search_runs()

In [0]:
best_run = runs.sort_values("metrics.rmse").iloc[0]

best_model = best_run["params.model_type"]
best_rmse = best_run["metrics.rmse"]

print("Best Model Selected")
print("Model:", best_model)
print("RMSE:", round(best_rmse, 2))

Best Model Selected
Model: LinearRegression
RMSE: 129.47
